# MindWave — Test Aggregation Report

This notebook runs all test suites across the MindWave project and produces a
consolidated pass/fail summary suitable for thesis appendix inclusion.

## Modules Tested

| Module | Framework | Tests |
|--------|-----------|-------|
| ML pipeline | pytest | windowing, features, scaler, LOSO, TFLite parity, FL simulation |
| Server API | pytest + httpx | auth, model API, k-anonymity, audit, privacy |
| Mobile (unit) | JUnit 4 | FeatureExtractor, ScalerNormalizer, XAI grouping |
| Wear (unit) | JUnit 4 | DataMap schema, buffer flush |


In [ ]:
import subprocess
import os
from pathlib import Path
from IPython.display import display, Markdown

REPO_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd())
if REPO_ROOT.name == 'ml':
    REPO_ROOT = REPO_ROOT.parent
print(f"Repository root: {REPO_ROOT}")

## 1. ML Pipeline Tests

In [ ]:
ml_result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    cwd=str(REPO_ROOT / "ml"),
    capture_output=True, text=True
)
print(ml_result.stdout)
if ml_result.returncode != 0:
    print("STDERR:", ml_result.stderr)

## 2. Server API Tests

In [ ]:
server_result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    cwd=str(REPO_ROOT / "server"),
    capture_output=True, text=True
)
print(server_result.stdout)
if server_result.returncode != 0:
    print("STDERR:", server_result.stderr)

## 3. Mobile & Wear Unit Tests

Run via Gradle. Requires Android SDK.

In [ ]:
gradle_cmd = "gradlew.bat" if os.name == "nt" else "./gradlew"
mobile_result = subprocess.run(
    [gradle_cmd, ":mobile:testDebugUnitTest", "--no-daemon"],
    cwd=str(REPO_ROOT),
    capture_output=True, text=True
)
print(mobile_result.stdout[-3000:] if len(mobile_result.stdout) > 3000 else mobile_result.stdout)
if mobile_result.returncode != 0:
    print("STDERR (last 2000):", mobile_result.stderr[-2000:])

In [ ]:
wear_result = subprocess.run(
    [gradle_cmd, ":wear:testDebugUnitTest", "--no-daemon"],
    cwd=str(REPO_ROOT),
    capture_output=True, text=True
)
print(wear_result.stdout[-3000:] if len(wear_result.stdout) > 3000 else wear_result.stdout)
if wear_result.returncode != 0:
    print("STDERR (last 2000):", wear_result.stderr[-2000:])

## 4. Summary Table

In [ ]:
def status_emoji(returncode: int) -> str:
    return "✅ PASS" if returncode == 0 else "❌ FAIL"

summary = f"""
| Module | Status |
|--------|--------|
| ML Pipeline | {status_emoji(ml_result.returncode)} |
| Server API | {status_emoji(server_result.returncode)} |
| Mobile Unit | {status_emoji(mobile_result.returncode)} |
| Wear Unit | {status_emoji(wear_result.returncode)} |
"""
display(Markdown(summary))

all_pass = all(r.returncode == 0 for r in [ml_result, server_result, mobile_result, wear_result])
print(f"\n{'='*50}")
print(f"OVERALL: {'ALL TESTS PASSED ✅' if all_pass else 'SOME TESTS FAILED ❌'}")
print(f"{'='*50}")

## 5. Test Coverage (Optional)

If `pytest-cov` is installed, re-run with coverage:

In [ ]:
# ML coverage
ml_cov = subprocess.run(
    ["python", "-m", "pytest", "tests/", "--cov=src", "--cov-report=term-missing", "--no-header"],
    cwd=str(REPO_ROOT / "ml"),
    capture_output=True, text=True
)
print(ml_cov.stdout[-4000:])